# 🌍 Landslide Detection — Full Pipeline
### Backend (CNN Training) + Frontend (Gradio UI)
**Dataset:** Landslide4Sense | **Model:** Lightweight U-Net CNN | **Input:** 14-channel H5 satellite images

---
## STEP 1 — Install Dependencies

In [7]:
# ── MASTER SETUP — run this first every session ───────────────────────────
import os, h5py, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from tensorflow.keras.models import Model

IMAGE_PATH = "landslide4sense/TrainData/img/"
MASK_PATH  = "landslide4sense/TrainData/mask/"
MODEL_SAVE = "landslide_model.keras"
BATCH_SIZE, EPOCHS, IMG_SIZE, N_CHANNELS = 4, 10, 128, 14
print("✅ Master setup complete")

✅ Master setup complete


In [8]:
# Run once — restart kernel after this cell completes
%pip install tensorflow h5py opencv-python scikit-learn matplotlib gradio opendatasets --quiet

Note: you may need to restart the kernel to use updated packages.


---
## STEP 2 — Download Dataset
> **Note:** You need a Kaggle account. Place your `kaggle.json` API key in `~/.kaggle/kaggle.json`  
> Or use opendatasets which will prompt for your Kaggle username + key.

In [9]:
import opendatasets as od

# This will prompt: Enter your Kaggle username and key
od.download("https://www.kaggle.com/datasets/tekbahadurkshetri/landslide4sense/data")

Skipping, found downloaded files in ".\landslide4sense" (use force=True to force download)


---
## STEP 3 — Imports & Config

In [10]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from tensorflow.keras.models import Model

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))

# ── Paths ──────────────────────────────────────────────────────────────────
# Update these paths if your dataset is extracted somewhere else
BASE_PATH   = "landslide4sense/TrainData/"
IMAGE_PATH  = os.path.join(BASE_PATH, "img/")
MASK_PATH   = os.path.join(BASE_PATH, "mask/")
MODEL_SAVE  = "landslide_model.keras"

BATCH_SIZE  = 4
EPOCHS      = 10
IMG_SIZE    = 128
N_CHANNELS  = 14

TensorFlow version: 2.21.0
GPU available: False


---
## STEP 4 — Verify Dataset Files

In [11]:
# STEP 4 — Verify Dataset Files
import os
import h5py
import numpy as np

# ── Paths (update if needed) ──────────────────────────────────────────────
BASE_PATH = "landslide4sense/TrainData/"
if not os.path.isdir(BASE_PATH):
    print(f"⚠️ BASE_PATH not found: {BASE_PATH!r}")
    BASE_PATH = None
    for root, dirs, files in os.walk("."):
        if "img" in dirs and "mask" in dirs:
            candidate = os.path.join(root, "")
            image_dir = os.path.join(candidate, "img")
            mask_dir = os.path.join(candidate, "mask")
            if os.path.isdir(image_dir) and os.path.isdir(mask_dir):
                BASE_PATH = candidate
                print(f"✅ Found dataset at: {BASE_PATH}")
                break

    if BASE_PATH is None:
        print(
            "Could not find the Landslide4Sense dataset folder. "
            "Make sure the dataset is downloaded and extracted, then rerun this cell."
        )
        print("Skipping dataset verification for now.")

if BASE_PATH is not None:
    IMAGE_PATH = os.path.join(BASE_PATH, "img/")
    MASK_PATH  = os.path.join(BASE_PATH, "mask/")
    MODEL_SAVE = "landslide_model.keras"

    BATCH_SIZE = 4
    EPOCHS     = 10
    IMG_SIZE   = 128
    N_CHANNELS = 14

    # ── Verify files ──────────────────────────────────────────────────────────
    image_files = sorted(os.listdir(IMAGE_PATH))
    mask_files  = sorted(os.listdir(MASK_PATH))

    assert len(image_files) == len(mask_files), "Mismatch between image and mask counts!"
    print(f"✅ Total images : {len(image_files)}")
    print(f"✅ Total masks  : {len(mask_files)}")

    # Peek at one sample to confirm shape
    with h5py.File(os.path.join(IMAGE_PATH, image_files[0]), 'r') as hf:
        sample_img = hf['img'][:]
    with h5py.File(os.path.join(MASK_PATH, mask_files[0]), 'r') as hf:
        sample_mask = hf['mask'][:]

    print(f"Sample image shape : {sample_img.shape}")   # Expected: (128, 128, 14)
    print(f"Sample mask shape  : {sample_mask.shape}")  # Expected: (128, 128)

⚠️ BASE_PATH not found: 'landslide4sense/TrainData/'
Could not find the Landslide4Sense dataset folder. Make sure the dataset is downloaded and extracted, then rerun this cell.
Skipping dataset verification for now.


---
## STEP 5 — Train / Test Split

In [12]:
from sklearn.model_selection import train_test_split

def find_landslide_dataset():
    search_roots = [
        ".",
        os.path.expanduser("~"),
        os.path.join(os.path.expanduser("~"), "Downloads"),
    ]
    for root_dir in search_roots:
        if not os.path.isdir(root_dir):
            continue
        for root, dirs, files in os.walk(root_dir):
            if "img" in dirs and "mask" in dirs:
                return root
    return None


    BASE_PATH = os.path.abspath(candidate)
    IMAGE_PATH = os.path.join(BASE_PATH, "img")
    MASK_PATH = os.path.join(BASE_PATH, "mask")
    print(f"✅ Found dataset at: {BASE_PATH}")

IMAGE_PATH = "landslide4sense/TrainData/img/"   # ← adjust this
MASK_PATH  = "landslide4sense/TrainData/mask/"  # ← adjust this

---
## STEP 6 — Memory-Efficient Data Generator

In [13]:
def data_generator(img_list, mask_list, batch_size=4):
    """Yields (images, masks) batches. Loops forever for Keras fit()."""
    while True:
        for i in range(0, len(img_list), batch_size):
            batch_img_files  = img_list[i : i + batch_size]
            batch_mask_files = mask_list[i : i + batch_size]

            images, masks = [], []

            for img_file, mask_file in zip(batch_img_files, batch_mask_files):
                with h5py.File(os.path.join(IMAGE_PATH, img_file), 'r') as hf:
                    img = hf['img'][:].astype(np.float32)
                with h5py.File(os.path.join(MASK_PATH, mask_file), 'r') as hf:
                    mask = hf['mask'][:].astype(np.float32)

                # Normalise image to [0, 1]
                max_val = np.max(img)
                if max_val > 0:
                    img = img / max_val

                mask = np.expand_dims(mask, axis=-1)  # (H, W) → (H, W, 1)

                images.append(img)
                masks.append(mask)

            yield np.array(images), np.array(masks)


---
## STEP 7 — Visualise a Sample

In [14]:
import os, h5py, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

IMAGE_PATH = "landslide4sense/TrainData/img/"
MASK_PATH  = "landslide4sense/TrainData/mask/"


In [15]:
import matplotlib.pyplot as plt
import numpy as np

# ── Get one sample batch ──────────────────────────────────────────────────
gen = data_generator(train_imgs, train_masks, batch_size=1)
sample_images, sample_masks = next(gen)

print(f"Image batch shape : {sample_images.shape}")   # Expected: (1, 128, 128, 14)
print(f"Mask batch shape  : {sample_masks.shape}")    # Expected: (1, 128, 128, 1)

# ── Pick a valid band to display (14 bands: index 0–13) ──────────────────
band = 3
img_display  = sample_images[0][:, :, band]
mask_display = sample_masks[0][:, :, 0]

# Clip to valid range for imshow
img_display  = np.clip(img_display, 0, 1)
mask_display = np.clip(mask_display, 0, 1)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(img_display, cmap='gray')
axes[0].set_title(f"Satellite Image (Band {band})")
axes[0].axis('off')

axes[1].imshow(mask_display, cmap='Reds')
axes[1].set_title("Ground Truth Mask")
axes[1].axis('off')

plt.tight_layout()
plt.savefig("sample_preview.png", dpi=100)
plt.show()
print("✅ Preview saved → sample_preview.png")

NameError: name 'train_imgs' is not defined

---
## STEP 8 — Build Lightweight U-Net Model

In [16]:
def build_model(img_size=128, n_channels=14):
    """Lightweight U-Net for binary landslide segmentation."""
    inputs = Input((img_size, img_size, n_channels))

    # ── Encoder ──────────────────────────────────────────────────────────
    c1 = Conv2D(16, 3, activation='relu', padding='same')(inputs)
    p1 = MaxPooling2D()(c1)                                        # 64×64

    c2 = Conv2D(32, 3, activation='relu', padding='same')(p1)
    p2 = MaxPooling2D()(c2)                                        # 32×32

    # ── Bottleneck ───────────────────────────────────────────────────────
    c3 = Conv2D(64, 3, activation='relu', padding='same')(p2)      # 32×32

    # ── Decoder ──────────────────────────────────────────────────────────
    u1 = UpSampling2D()(c3)                                        # 64×64
    u1 = concatenate([u1, c2])

    u2 = UpSampling2D()(u1)                                        # 128×128
    u2 = concatenate([u2, c1])

    outputs = Conv2D(1, 1, activation='sigmoid')(u2)

    model = Model(inputs, outputs, name="LightweightUNet")
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


model = build_model(IMG_SIZE, N_CHANNELS)
model.summary()

Model: "LightweightUNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 14)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │      2,032 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 64, 64,    │          0 │ conv2d_2[0][0]    │
│ (UpSampling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64, 64,    │          0 │ up_sampling2d[0]… │
│ (Concatenate)       │ 96)               │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 128, 128,  │          0 │ concatenate[0][0] │
│ (UpSampling2D)      │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 128, 128,  │          0 │ up_sampling2d_1[… │
│ (Concatenate)       │ 112)              │            │ conv2d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │        113 │ concatenate_1[0]… │
│                     │ 1)                │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 25,281 (98.75 KB)

 Trainable params: 25,281 (98.75 KB)

 Non-trainable params: 0 (0.00 B)

---
## STEP 9 — Train

In [17]:
steps_per_epoch = max(1, len(train_imgs) // BATCH_SIZE)

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    verbose=1
)

print("\n✅ Training complete")

NameError: name 'train_imgs' is not defined

---
## STEP 10 — Evaluate & Plot Metrics

In [18]:
test_steps = max(1, len(test_imgs) // BATCH_SIZE)
loss, accuracy = model.evaluate(test_gen, steps=test_steps, verbose=0)
print(f"Test Loss     : {loss:.4f}")
print(f"Test Accuracy : {accuracy:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], color='steelblue', linewidth=2)
ax1.set_title("Training Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], color='tomato', linewidth=2)
ax2.set_title("Training Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=100)
plt.show()

NameError: name 'test_imgs' is not defined

---
## STEP 11 — Visualise Predictions

In [19]:
test_images, test_masks_batch = next(data_generator(test_imgs, test_masks, batch_size=4))
predictions = model.predict(test_images, verbose=0)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for i in range(4):
    axes[i, 0].imshow(test_images[i][:, :, 3], cmap='gray')
    axes[i, 0].set_title("Satellite (Band 3)")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(test_masks_batch[i][:, :, 0], cmap='Reds')
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(predictions[i][:, :, 0], cmap='Reds', vmin=0, vmax=1)
    axes[i, 2].set_title("Predicted Mask")
    axes[i, 2].axis('off')

plt.suptitle("Landslide Segmentation Results", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("prediction_results.png", dpi=100, bbox_inches='tight')
plt.show()

NameError: name 'test_imgs' is not defined

---
## STEP 12 — Save Model

In [20]:
import os, h5py, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate
from tensorflow.keras.models import Model

IMAGE_PATH = "landslide4sense/TrainData/img/"
MASK_PATH  = "landslide4sense/TrainData/mask/"
MODEL_SAVE = "landslide_model.keras"
BATCH_SIZE = 4
EPOCHS     = 10
IMG_SIZE   = 128
N_CHANNELS = 14
print("✅ Imports & config done")

✅ Imports & config done


In [21]:
model.save(MODEL_SAVE)
print(f"✅ Model saved → {MODEL_SAVE}")

✅ Model saved → landslide_model.keras


---
## STEP 13 — Frontend: Gradio Web App
Launches a local web UI at **http://127.0.0.1:7860**  
Upload any `.h5` image file from the dataset and see the predicted landslide mask.

In [27]:
import gradio as gr
import tensorflow as tf
import h5py, numpy as np, matplotlib, matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
import tempfile, os

matplotlib.use('Agg')  # Non-interactive backend for Gradio

# ── Load model (uses the model trained above, or loads from disk) ─────────
try:
    _model = model   # already in memory from training above
    print("Using in-memory trained model.")
except NameError:
    _model = tf.keras.models.load_model(MODEL_SAVE)
    print(f"Loaded model from {MODEL_SAVE}")


def predict_landslide(h5_file):
    """
    Accepts an uploaded .h5 image file.
    Returns a 3-panel matplotlib figure: satellite | ground truth (if available) | prediction.
    """
    if h5_file is None:
        return None, "⚠️ Please upload an H5 image file."

    try:
        with h5py.File(h5_file.name, 'r') as hf:
            keys = list(hf.keys())
            if 'img' not in keys:
                return None, f"❌ Key 'img' not found. File contains: {keys}"
            img = hf['img'][:].astype(np.float32)
            has_mask = 'mask' in keys
            gt_mask  = hf['mask'][:] if has_mask else None

        # Normalise
        max_val = np.max(img)
        if max_val > 0:
            img = img / max_val

        # Predict
        inp        = np.expand_dims(img, axis=0)   # (1, 128, 128, 14)
        prediction = _model.predict(inp, verbose=0)[0, :, :, 0]  # (128, 128)
        binary_mask = (prediction > 0.5).astype(np.uint8)

        landslide_pct = binary_mask.mean() * 100
        status = f"🔴 Landslide detected ({landslide_pct:.1f}% area)" if landslide_pct > 1 else "🟢 No significant landslide detected"

        # ── Plot ─────────────────────────────────────────────────────────
        ncols = 3 if has_mask else 2
        fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

        axes[0].imshow(img[:, :, 3], cmap='gray')
        axes[0].set_title("Satellite Image\n(Band 3)", fontsize=12)
        axes[0].axis('off')

        if has_mask:
            axes[1].imshow(gt_mask, cmap='Reds')
            axes[1].set_title("Ground Truth Mask", fontsize=12)
            axes[1].axis('off')
            pred_ax = axes[2]
        else:
            pred_ax = axes[1]

        pred_ax.imshow(prediction, cmap='Reds', vmin=0, vmax=1)
        pred_ax.set_title("Predicted Landslide Mask", fontsize=12)
        pred_ax.axis('off')

        plt.suptitle(status, fontsize=13, color='darkred' if landslide_pct > 1 else 'darkgreen', y=1.02)
        plt.tight_layout()

        buf = BytesIO()
        plt.savefig(buf, format='png', dpi=120, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        result_img = Image.open(buf).copy()
        buf.close()

        return result_img, status

    except Exception as e:
        return None, f"❌ Error: {str(e)}"


# ── Build Gradio Interface ────────────────────────────────────────────────
with gr.Blocks(title="Landslide Detection", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🌍 Landslide Detection System
    Upload a `.h5` satellite image file from the **Landslide4Sense** dataset.  
    The model will predict which regions are affected by landslides.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(
                label="Upload H5 Image File (.h5)",
                file_types=[".h5", ".hdf5"]
            )
            predict_btn = gr.Button("🔍 Detect Landslide", variant="primary", size="lg")
            status_box  = gr.Textbox(label="Detection Result", interactive=False)

        with gr.Column(scale=2):
            output_img = gr.Image(label="Prediction Output", type="pil")

    gr.Markdown("""
    ---
    **Model:** Lightweight U-Net CNN  
    **Input:** 14-channel satellite imagery (128×128 px)  
    **Output:** Binary segmentation mask (red = landslide)
    """)

    predict_btn.click(
        fn=predict_landslide,
        inputs=[file_input],
        outputs=[output_img, status_box]
    )

# ── Launch ────────────────────────────────────────────────────────────────
demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    share=False,   # Set share=True for a public URL
    inbrowser=True
)

Using in-memory trained model.


C:\Users\AMRETA\AppData\Local\Temp\ipykernel_26692\3264733491.py:86: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Landslide Detection", theme=gr.themes.Soft()) as demo:


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.